# 08 · Reranking: the second stage of candidate evaluation

The retriever quickly gathers a broad pool. The reranker reads the **question and the candidate text together**, evaluates whether the fragment actually answers the query, and orders the short list. We will learn about cross-encoders, late interaction, generative LLMs, and point-wise decision models. We will run the local **System One Clef Flash and Clef** via Ollama. These are decision models; do not call them via `chat`.

The Clef result is a judgment of relevance, not a probability of truth. Fact verification still requires the original fragment and its metadata.

In [ ]:
# Only cell with imports and project path.
import os
import re
import sys
import time
from pathlib import Path

import numpy as np
from dotenv import dotenv_values
from ollama import Client
from rank_bm25 import BM25Okapi

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import embed_texts, ollama_settings, section_records

## 1. Candidates from two cheap search engines

We build a small RRF as in lesson 07, but the notebook works independently. The reranker receives only the top-6; costs increase with the number and length of the fragments. Long input may be truncated by the model, so we provide explicitly limited text and do not treat the score as evidence for the unseen remainder.

In [ ]:
records = section_records()
question = "Why is a percentage forecast error undefined when demand is zero?"
bm25 = BM25Okapi([re.findall("[a-z0-9]+", row["text"].lower()) for row in records])
lexical = bm25.get_scores(re.findall("[a-z0-9]+", question.lower()))
vectors = embed_texts([row["text"] for row in records])
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
q = embed_texts([question])[0]
q /= np.linalg.norm(q)
dense = vectors @ q
fused = np.zeros(len(records))
for ranking in (np.argsort(-lexical)[:10], np.argsort(-dense)[:10]):
    for rank, index in enumerate(ranking, 1):
        fused[index] += 1 / (60 + rank)
candidate_indices = np.argsort(-fused)[:6]
candidates = [records[int(i)] for i in candidate_indices]
[(row["id"], row["heading"]) for row in candidates]

## 2. System One: one score-type question for the candidate

`state` contains the question and explicitly numbered texts. For each ID, we pass criteria 0–3: irrelevant, related topic, partial answer, direct answer. The API returns a distribution of scores and a **weighted score**; we sort it in descending order. We do not call it "confidence that the fragment is true."

Clef Flash is a faster variant; Clef is larger. Runs are independent, and names can be set via `RAG_RERANK_MODELS` in the root `.env` (comma-separated list). If local tags are old, `ollama pull clef-flash:latest` and `ollama pull clef:latest` will refresh the System One format.

In [ ]:
def rerank_with_systemone(model, question, candidates):
    state = {"query": question, "passages": [{"id": row["id"], "text": row["text"][:1100]} for row in candidates]}
    questions = {
        f"p{i}": {"type": "score", "instructions": f"Rate only passage {i} for whether it directly answers the query; ignore other passages.",
                 "criteria": ["Unrelated", "Same broad topic but no answer", "Partly answers", "Directly answers with relevant evidence"]}
        for i in range(len(candidates))
    }
    response = Client(host=ollama_settings()[0]).systemone(model=model, state=state, questions=questions)
    return sorted([{"id": row["id"], "score_0_to_3": float(response.answers[f"p{i}"].score)}
                   for i, row in enumerate(candidates)], key=lambda row: -row["score_0_to_3"])


config = dotenv_values(ROOT / ".env")
model_names = (os.environ.get("RAG_RERANK_MODELS") or config.get("RAG_RERANK_MODELS") or "clef-flash:latest,clef:latest").split(",")
comparisons = {}
for model in model_names:
    start = time.perf_counter()
    comparisons[model.strip()] = {"ranking": rerank_with_systemone(model.strip(), question, candidates),
                                  "seconds": round(time.perf_counter() - start, 2)}
comparisons

## 3. Read the difference, not just the new list

A reranker might move the correct fragment from position 4 to 1, but it can also make a mistake. Compare the top-1 result of both models against the source text. When the results differ, it is a signal for inspection, not for automatic voting. Evaluate `nDCG@k`, candidate recall, latency, and cost on manually labeled questions.

In [ ]:
original = [row["id"] for row in candidates]
{"retriever_top3": original[:3],
 "reranked_top3": {model: [item["id"] for item in data["ranking"][:3]] for model, data in comparisons.items()},
 "first_source_excerpt": next(row["text"][:350] for row in records if row["id"] == next(iter(comparisons.values()))["ranking"][0]["id"])}

### Reranker Types

| Type | What it sees | Pros | Cost / Limitation |
|---|---|---|---|
| Cross-encoder | query and one fragment at a time | precise pair assessment | cost for every candidate |
| Late interaction (e.g., ColBERT) | token representations of both sides | more accurate token matching | larger index and complexity |
| Generative LLM | a list or pairs in the prompt | can be asked for reasoning | time, hallucinations, unstable format |
| Clef System One | state text and explicit scoring criteria | assessment of multiple fragments in one call | evaluation depends on criteria and input limit |

**Exercise:** change the question to one that the corpus does not support. Does even the highest score mean sufficient evidence? Add a separate rejection condition or a sufficiency score. The [Official Clef card](https://ollama.com/library/clef) describes the `systemone` API, the `score` type, and the meaning of `confidence`.